# Importar librerias

In [ ]:
!pip install optuna

In [ ]:
import pandas as pd
import json
import re
import numpy as np
import torch
import os
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
import xgboost as xgb
import optuna


from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer
from PIL import Image
from sklearn.model_selection import train_test_split
from collections import Counter

# Carga de datos

In [ ]:
# directorio del json
PATH_JSON = '/content/drive/MyDrive/EXIST 2026 Memes Dataset/training/EXIST2026_training.json'

# Cargo el json
print("Cargando el dataset EXIST 2026...")
try:
    with open(PATH_JSON, 'r', encoding='utf-8') as f:
        mis_datos_json = json.load(f)
    print(f"¡Éxito! Se han cargado {len(mis_datos_json)} memes correctamente.")
except FileNotFoundError:
    print("Error: No se ha encontrado el archivo.")

def extraer_datos_por_usuario(datos_json):
    filas = []

    # Itero sobre cada meme en el JSON
    for id_meme, info in datos_json.items():

        # Extraigo la información base del meme
        base_info = {
            'id': info.get('id_EXIST'),
            'lang': info.get('lang'),
            'text': info.get('text'),
            'meme': info.get('meme'),
            'number_annotators': info.get('number_annotators')
        }

        # Verifico si este meme tiene datos sensoriales
        if 'sensorial' in info and 'users' in info['sensorial']:
            lista_usuarios = info['sensorial']['users']
            modalidades = info['sensorial'].get('modalities', {})

            # Creo UNA FILA por cada usuario
            for usuario in lista_usuarios:
                fila_usuario = base_info.copy()
                fila_usuario['sensor_user'] = usuario # Guardo quién es el usuario

                # Busco las métricas de este usuario en TODAS las modalidades (ET, HR, EEG)
                for modalidad_nombre, modalidad_datos in modalidades.items():
                    if 'by_user' in modalidad_datos and usuario in modalidad_datos['by_user']:
                        metricas_usuario = modalidad_datos['by_user'][usuario]

                        # Añado cada sensor como una columna nueva
                        for metrica, valor in metricas_usuario.items():
                            # Guardo la métrica.
                            nombre_columna = f"{modalidad_nombre}_{metrica}"
                            fila_usuario[nombre_columna] = valor

                # Añado la fila de este usuario a la lista general
                filas.append(fila_usuario)

    # Convierto la lista de diccionarios en un DataFrame de Pandas
    df = pd.DataFrame(filas)
    return df

# Ejecuto la función
print("Transformando JSON a formato tabular...")
df_sensores = extraer_datos_por_usuario(mis_datos_json)

# Muestro el resultado
print(f"¡Hecho! El nuevo dataset tiene {df_sensores.shape[0]} filas y {df_sensores.shape[1]} columnas.")
display(df_sensores.head(3))

In [ ]:
# Añado los labels
print("Añadiendo labels...")
mapeo_etiquetas = {}
for id_meme, info in mis_datos_json.items():
    votos = info.get('labels_task2_1', [])
    yes_count = votos.count('YES')
    no_count = votos.count('NO')

    if yes_count > no_count:
        mapeo_etiquetas[id_meme] = 'YES'
    elif no_count > yes_count:
        mapeo_etiquetas[id_meme] = 'NO'
    else:
        mapeo_etiquetas[id_meme] = 'EMPATE'

# Añado la nueva columna a df_sensores
df_sensores['label_task1'] = df_sensores['id'].map(mapeo_etiquetas)

# Imprimo un resultado de ejemplo
print("EJEMPLO DE LAS PRIMERAS FILAS CON LA NUEVA COLUMNA:")
display(df_sensores[['id', 'sensor_user', 'label_task1']].head(5))

# recuento inicial
print("\nRECUENTO INICIAL (YES, NO, EMPATE):")
print(df_sensores['label_task1'].value_counts())

# Borro todas las filas que sean 'EMPATE'
print("\nBorrando los casos de 'EMPATE'...")
df_sensores = df_sensores[df_sensores['label_task1'] != 'EMPATE'].copy()

# Vuelvo a sacar el recuento final
print("\nRECUENTO FINAL (SIN EMPATES):")
print(df_sensores['label_task1'].value_counts())

print(f"\nListo, df_sensores ahora tiene {df_sensores.shape[0]} filas.")
display(df_sensores.head(5))

limpieza de texto

In [ ]:
# Función de limpieza
def limpiar_texto_meme(texto):
    if not isinstance(texto, str):
        return ""

    # Elimino URLs (http, https, www)
    texto = re.sub(r'http\S+|www\.\S+|\b[\w-]+\.(com|es|net|org|co)\b\S*', '', texto)

    # Elimino menciones (@usuario) y hashtags vacíos
    texto = re.sub(r'@\w+', '', texto)

    # Cambio guiones por espacios (Arregla cosas tipo "COMO-QUE-NO")
    texto = re.sub(r'-', ' ', texto)

    # Quito caracteres especiales muy raros pero dejar puntuación básica (!?.)
    texto = re.sub(r'[^\w\s\.,!?¡¿]', ' ', texto)

    # Reemplazo múltiples espacios o saltos de línea por un solo espacio
    texto = re.sub(r'\s+', ' ', texto)

    # Quito espacios al principio y al final
    return texto.strip()

#Aplico la limpueza
print("Limpiando el texto de los memes...")

# Guardo el texto original un momento para poder comparar
texto_original_ejemplo = df_sensores['text'].iloc[0]

# Aplico la función a toda la columna
df_sensores['text'] = df_sensores['text'].apply(limpiar_texto_meme)

# Comprobación
print("\nCOMPROBACIÓN DEL ANTES Y DESPUÉS:")
print("-" * 50)
print(f"ANTES:   {texto_original_ejemplo}")
print(f"DESPUÉS: {df_sensores['text'].iloc[0]}")
print("-" * 50)
print("Limpieza completada.")

corrección de valores NaN

In [ ]:
# Selecciono solo las columnas numéricas
cols_numericas = df_sensores.select_dtypes(include=['float64', 'int64']).columns.tolist()
cols_sensores = [c for c in cols_numericas if c not in ['number_annotators']]

print(f"Se han identificado {len(cols_sensores)} columnas de sensores para imputar.")

# Cuento cuántos NaN hay antes de empezar
nans_antes = df_sensores[cols_sensores].isna().sum().sum()
print(f"Valores NaN totales ANTES de la imputación: {nans_antes}")

# Agrupamos por 'id' (el meme) y relleno los NaN con la media de ese grupo
df_sensores[cols_sensores] = df_sensores.groupby('id')[cols_sensores].transform(lambda x: x.fillna(x.mean()))

# Si un grupo entero era NaN, el paso anterior los deja como NaN.
# Los relleno con la mediana global de la columna entera.
df_sensores[cols_sensores] = df_sensores[cols_sensores].fillna(df_sensores[cols_sensores].median())

# Compruebo resultados
nans_despues = df_sensores[cols_sensores].isna().sum().sum()
print(f"Valores NaN totales DESPUÉS de la imputación: {nans_despues}")

if nans_despues == 0:
    print("¡DataFrame limpio!")
display(df_sensores.head(5))

Columna sobre anotadores

In [ ]:
def crear_vector_anotadores(datos_json):
    filas_demografia = []

    for id_meme, info in datos_json.items():
        # Extraigo las listas del JSON
        genders = info.get('gender_annotators', [])
        ages = info.get('age_annotators', [])
        ethnicities = info.get('ethnicities_annotators', [])
        studies = info.get('study_levels_annotators', [])
        countries = info.get('countries_annotators', [])

        num_annotators = len(genders)
        if num_annotators == 0:
            continue

        # Diccionario base para este meme
        perfil = {'id': id_meme}

        # Función auxiliar para contar y sacar el porcentaje
        def calcular_proporciones(lista_rasgos, prefijo):
            conteos = Counter(lista_rasgos)
            for rasgo, cantidad in conteos.items():
                # Normalizo nombres y creo la columna
                nombre_col = f"{prefijo}_{rasgo.replace(' ', '_').replace('-', '_')}"
                perfil[nombre_col] = cantidad / num_annotators

        # Calculo proporciones para todas las categorías
        calcular_proporciones(genders, 'dem_gender')
        calcular_proporciones(ages, 'dem_age')
        calcular_proporciones(ethnicities, 'dem_eth')
        calcular_proporciones(studies, 'dem_study')
        calcular_proporciones(countries, 'dem_country')

        filas_demografia.append(perfil)

    # Convierto a DataFrame y rellenamos con 0 los rasgos que no aparezcan en un meme
    df_demo = pd.DataFrame(filas_demografia).fillna(0.0)
    return df_demo

#Uno los DF
print("Calculando el Vector Demográfico de los anotadores...")
df_demografia = crear_vector_anotadores(mis_datos_json)

print(f"Se han generado {df_demografia.shape[1] - 1} nuevas columnas de contexto demográfico.")

# Hago un MERGE con df_sensores usando la columna 'id'
df_sensores = pd.merge(df_sensores, df_demografia, on='id', how='left')

# Verifico que no se hayan creado NaN en el merge y relleno con 0 si los hay
cols_demograficas = [c for c in df_sensores.columns if c.startswith('dem_')]
df_sensores[cols_demograficas] = df_sensores[cols_demograficas].fillna(0.0)

# Resultado
print("\nAsí se ve el contexto para el primer meme:")
display(df_sensores[['id'] + cols_demograficas].head(1))
display(df_sensores.head(5))

Division en train, val y test

In [ ]:
print("Iniciando división segura del dataset...")

# Aseguro de que la etiqueta es numérica (0 y 1)
if 'label' not in df_sensores.columns:
    df_sensores['label'] = df_sensores['label_task1'].map({'YES': 1, 'NO': 0})

# Saco los IDs únicos y su etiqueta para hacer el split equilibrado
df_ids_unicos = df_sensores.drop_duplicates(subset=['id'])[['id', 'label']]

# Divido los IDs (80% Train, 10% Val, 10% Test)
ids_train, ids_temp = train_test_split(df_ids_unicos['id'], test_size=0.20, random_state=42, stratify=df_ids_unicos['label'])
ids_val, ids_test = train_test_split(ids_temp, test_size=0.50, random_state=42, stratify=df_ids_unicos[df_ids_unicos['id'].isin(ids_temp)]['label'])

# Filtro el dataset completo usando esos IDs
df_train_xgb = df_sensores[df_sensores['id'].isin(ids_train)].copy()
df_val_xgb = df_sensores[df_sensores['id'].isin(ids_val)].copy()
df_test_xgb = df_sensores[df_sensores['id'].isin(ids_test)].copy()

# XGBoost solo traga números. Quito textos, nombres de archivo, etc.
columnas_a_quitar = ['id', 'meme', 'text', 'lang', 'label_task1', 'sensor_user']

# Me quedo solo con la Fisiología, la Demografía, la predicción de mCLIP y el Label
X_train_xgb = df_train_xgb.drop(columns=columnas_a_quitar + ['label'])
y_train_xgb = df_train_xgb['label']

X_val_xgb = df_val_xgb.drop(columns=columnas_a_quitar + ['label'])
y_val_xgb = df_val_xgb['label']

X_test_xgb = df_test_xgb.drop(columns=columnas_a_quitar + ['label'])
y_test_xgb = df_test_xgb['label']

print("\n¡División completada!")
print("-" * 40)
print(f"Train: {X_train_xgb.shape[0]} registros fisiológicos (Características: {X_train_xgb.shape[1]})")
print(f"Val:   {X_val_xgb.shape[0]} registros fisiológicos")
print(f"Test:  {X_test_xgb.shape[0]} registros fisiológicos")
print("-" * 40)

In [ ]:
print("INICIANDO 'EL CONSEJO DE SABIOS' (5-Fold Ensembling)...")

# Unimos Train y Val para que el K-Fold tenga más datos de donde aprender
# (Mantenemos X_test_super intacto para la evaluación final, ¡no hacemos trampa!)
X_train_val = np.vstack((X_train_super_scaled, X_val_super_scaled))
y_train_val = np.concatenate((y_train_xgb.values, y_val_xgb.values))

#  Configuración del K-Fold (5 particiones)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
device = "cuda" if torch.cuda.is_available() else "cpu"
dimension_total = X_train_val.shape[1]

# Lista para guardar los 5 cerebros entrenados
modelos_ensamblados = []

# ENTRENAMIENTO DE LOS 5 MODELOS
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
    print(f"\nEntrenando Cerebro {fold + 1}/5...")

    # Preparamos los datos para este fold
    X_f_train, y_f_train = X_train_val[train_idx], y_train_val[train_idx]
    X_f_val, y_f_val = X_train_val[val_idx], y_train_val[val_idx]

    train_ds = TensorDataset(torch.FloatTensor(X_f_train), torch.LongTensor(y_f_train))
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

    # Instanciamos la arquitectura Early Fusion
    modelo = EarlyFusionNN(input_dim=dimension_total).to(device)
    optimizer = optim.AdamW(modelo.parameters(), lr=0.0005, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Entrenamos 20 épocas por modelo
    for epoch in range(20):
        modelo.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = modelo(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Guardamos el modelo en modo evaluación
    modelo.eval()
    modelos_ensamblados.append(modelo)
    print(f"Cerebro {fold + 1} listo.")

# ==========================================
# VOTACIÓN FINAL SOBRE EL CONJUNTO DE TEST
# ==========================================
print("\n" + "="*50)
print(" VOTACIÓN FINAL SOBRE EL TEST SET")
print("="*50)

#  test 
test_ds = TensorDataset(torch.FloatTensor(X_test_super_scaled))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

predicciones_totales = []

with torch.no_grad():
    for batch_X in test_loader:
        batch_X = batch_X[0].to(device)

        # Guardaremos las probabilidades de los 5 modelos
        batch_probs = torch.zeros((batch_X.size(0), 2)).to(device)

        # Hacemos que cada modelo vote
        for modelo in modelos_ensamblados:
            outputs = modelo(batch_X)
            probs = F.softmax(outputs, dim=1)
            batch_probs += probs 

        batch_probs /= 5.0
        preds = torch.argmax(batch_probs, dim=1)
        predicciones_totales.extend(preds.cpu().numpy())

# ==========================================
# RESULTADOS DEFINITIVOS
# ==========================================
print(classification_report(y_test_xgb.values, predicciones_totales, digits=4))

cm_ensamblado = confusion_matrix(y_test_xgb.values, predicciones_totales)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_ensamblado, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=['No Sexista (0)', 'Sexista (1)'],
            yticklabels=['No Sexista (0)', 'Sexista (1)'])
plt.xlabel('Predicción del Consejo de Sabios')
plt.ylabel('Etiqueta Real')
plt.title('Matriz de Confusión: Ensembling (5-Fold)')
plt.show()